# Infilling Score — frozen StarCoder2-3B, 128-token source-code adaptation
Raoof et al., ICLR 2025: [paper](https://proceedings.iclr.cc/paper_files/paper/2025/file/5709163243753c9c9ab7f4b4e5a8766d-Paper-Conference.pdf), [author implementation, pinned commit](https://github.com/neginraoof/infilling-score/tree/54ee252e378afffe09f4dbde5c2ca3adbf8ccfe0). The full method and source were inspected. This experiment implements one author-normalized Infilling Score from the Bayes-ratio derivation, five future tokens, bottom-20% arithmetic mean, higher = member. Appendix pseudocode differs from main equations/source; the README and research provenance document the precise interpretation and boundary indexing.

Frozen causal LM, no masked/bidirectional head, FIM prompt, reference model, training, score search or sweep. Analyze the first contiguous **128 baseline-tokenizer input IDs** in every language. Native special tokens count toward the span; the first input ID is context, never an invented BOS. All predictable, non-special/nonempty-offset targets in the span are considered (usually 127), including boundary targets with shortened/no future windows exactly as the pinned author code. Record every effective count and short/invalid span.

All original 1600/200/200/1000 manifests are validated first. Smoke selects 80/20/20/30 with seed 42; full preserves all saved rows. Only calibration selects the threshold. GPU inference runs manually on Kaggle; no original-paper benchmark equivalence claim.

In [ ]:
from pathlib import Path
INPUT_ROOT = Path('/kaggle/input')
WORK_ROOT = Path('/kaggle/working')
UPLOAD_SOURCE = INPUT_ROOT / 'EDIT-infilling-source-dataset'
UPLOAD_ZIP = UPLOAD_SOURCE / 'infilling_kaggle_source.zip'
PROJECT_ROOT = WORK_ROOT / 'infilling-code-source-v1'
BASELINE_RUN = ''  # discover one saved complete baseline run, or set explicitly
PATH_OVERRIDES = dict(shadow='',validation='',calibration='',heldout='',scores='',config='')
ADAPTIVE_SCORES = ''  # intended heldout_adaptive_scores.parquet; full scores recommended
MODE = 'smoke'  # 'smoke' or 'full'
SMOKE_SIZES = dict(shadow=80,validation=20,calibration=20,heldout=30)
RUN_ID = f'infilling-code-{MODE}-v1'
OUT = WORK_ROOT / 'infilling_outputs' / RUN_ID
RESUME_FROM = ''  # trusted saved OUT from the SAME mode/scientific identity
CACHE_REUSE_FROM = ''  # optional saved infilling_cache.sqlite, e.g. smoke -> full
MODEL_CACHE_DIR = ''  # empty inherits baseline HF model cache root
MODEL_LOCAL_FILES_ONLY = True
ALLOW_MODEL_DOWNLOAD = False  # True explicitly permits pinned model preparation on Kaggle
DATASET_CACHE_DIR = WORK_ROOT / 'hf_cache/datasets'
SOURCE_ACCESS = 'streaming'  # streaming | cached (existing Arrow cache only)
SESSION_BUDGET_MINUTES = 480
DISK_RESERVE_GIB = 3.
WORK_ROOT.mkdir(parents=True,exist_ok=True)
# Science is frozen in the verified reproducibility.json: seed 42, span 128,
# future tokens 5, bottom fraction .2, author normalization/boundaries.

## Verify the portable source package
Upload `infilling_kaggle_source.zip` as a Kaggle Dataset or attach its extracted contents and edit UPLOAD_SOURCE. This uses the working adaptive notebook's source bootstrap, pinned cache conventions and protected dependencies. Enable a Tesla T4 and Internet for streaming. Code is verified before imports. Existing attack implementations and outputs are not modified.

In [ ]:
import hashlib, json, os, shutil, subprocess, sys, zipfile
SOURCE_MANIFEST_SHA256 = '3139d45bba836b5d745696f3587fa4801028feea2a1d016cb54fa96a74879d39'
if not (PROJECT_ROOT / "infilling_code_attack/source_manifest.json").is_file():
    if PROJECT_ROOT.exists():
        raise FileExistsError("Incomplete PROJECT_ROOT; select a fresh directory")
    if (UPLOAD_SOURCE / "infilling_code_attack/source_manifest.json").is_file():
        source_manifest_path = UPLOAD_SOURCE / "infilling_code_attack/source_manifest.json"
        if hashlib.sha256(source_manifest_path.read_bytes()).hexdigest() != SOURCE_MANIFEST_SHA256:
            raise ValueError("Outdated Infilling Score source package")
        source_manifest = json.loads(source_manifest_path.read_text())
        PROJECT_ROOT.mkdir(parents=True)
        for name in list(source_manifest) + ["infilling_code_attack/source_manifest.json"]:
            destination = (PROJECT_ROOT / name).resolve()
            if not destination.is_relative_to(PROJECT_ROOT.resolve()):
                raise ValueError("Unsafe source path")
            destination.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(UPLOAD_SOURCE / name, destination)
    else:
        with zipfile.ZipFile(UPLOAD_ZIP) as archive:
            for entry in archive.infolist():
                if not (PROJECT_ROOT / entry.filename).resolve().is_relative_to(PROJECT_ROOT.resolve()):
                    raise ValueError("Unsafe archive path")
            zipped_manifest = archive.read('infilling_code_attack/source_manifest.json')
            if hashlib.sha256(zipped_manifest).hexdigest() != SOURCE_MANIFEST_SHA256:
                raise ValueError('Outdated Infilling Score source ZIP')
            allowed = set(json.loads(zipped_manifest)) | {'infilling_code_attack/source_manifest.json'}
            if set(archive.namelist()) != allowed:
                raise ValueError('Unexpected archive files')
            archive.extractall(PROJECT_ROOT)
manifest_path = PROJECT_ROOT / "infilling_code_attack/source_manifest.json"
if hashlib.sha256(manifest_path.read_bytes()).hexdigest() != SOURCE_MANIFEST_SHA256:
    raise ValueError("Source manifest mismatch; use a fresh PROJECT_ROOT and rebuilt package")
source_manifest = json.loads(manifest_path.read_text())
for name, expected in source_manifest.items():
    if hashlib.sha256((PROJECT_ROOT / name).read_bytes()).hexdigest() != expected:
        raise ValueError("Source mismatch: " + name)
sys.path.insert(0, str(PROJECT_ROOT))
UPSTREAM = PROJECT_ROOT / "SERSEM-Poisoned-Chalice-Competition-2026"
sys.path.insert(0, str(UPSTREAM / "code"))

In [ ]:
# Install the existing pinned baseline dependencies; preserve Kaggle's numerical stack.
from shared.kaggle_setup import install_dependencies, validate_environment
os.environ["USE_TF"] = "0"
os.environ["USE_FLAX"] = "0"
os.environ["HF_HOME"] = str(WORK_ROOT / "hf_cache")
os.environ["HF_DATASETS_CACHE"] = str(DATASET_CACHE_DIR)
os.environ["HF_HUB_DISABLE_XET"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
packages = ["libenchant-2-2", "hunspell-en-us"]
status = subprocess.run(["dpkg-query", "-W", "-f=${Status}\n", *packages], capture_output=True, text=True)
if status.returncode != 0 or status.stdout.splitlines().count("install ok installed") != len(packages):
    subprocess.run(["apt-get", "update", "-qq"], check=True)
    subprocess.run(["apt-get", "install", "-y", "-qq", *packages], check=True)
install_dependencies(PROJECT_ROOT / "shared/requirements-kaggle.txt", WORK_ROOT)
environment = validate_environment(PROJECT_ROOT / "shared/requirements-kaggle.txt")

In [ ]:
# Lightweight CPU checks: Bayes partition cancellation, normalized author score,
# causal indexing, padding, resume and leakage. No pretrained models/downloads.
subprocess.run([sys.executable,'-m','unittest','discover','-s','tests','-p','test_infilling.py','-v'],
               cwd=PROJECT_ROOT,check=True)

## Validate complete saved data and comparison coverage
No new splits. Both modes validate every original manifest first. Smoke selection is balanced by language/membership using seed 42. Its IDs may differ from notebook 03's smoke selection, so full saved adaptive scores are recommended; missing exact comparisons fail before inference. Full requires all 1000 adaptive held-out scores. Baseline numerical/API versions must match as in notebook 03.

In [ ]:
import copy, random
import numpy as np
import pandas as pd
from adaptive_token_attack.data import resolve_paths,load_run,select_partitions,frame_fingerprint
from adaptive_token_attack.features import load_frozen_target
from infilling_code_attack.cache import Cache,fingerprint
from infilling_code_attack.data import scoring_identity,prepare_plans,fixed_configuration
from infilling_code_attack.inference import TorchScorer
from infilling_code_attack.runner import run_scores
from infilling_code_attack.evaluation import exact_match,evaluate
from infilling_code_attack import METHOD_NAME,METHOD_STATUS
from shared.resource_pipeline import Monitor,restore_run,file_hash
from shared.resource_data import SourceStore
from shared.sersem_baseline import write_json
fixed = fixed_configuration()
print(METHOD_NAME, METHOD_STATUS)
paths = resolve_paths(BASELINE_RUN,PATH_OVERRIDES,INPUT_ROOT)
original,saved_scores,base_cfg = load_run(paths)
if SMOKE_SIZES != fixed['smoke_counts']:
    raise ValueError('Smoke counts are fixed to 80/20/20/30')
partitions,selection = select_partitions(original,MODE,SMOKE_SIZES,seed=fixed['seed'])
if set(pd.concat(list(original.values())).language) != {'Go','Java','Python','Ruby','Rust'}:
    raise ValueError('Unexpected saved programming languages')
if base_cfg['versions'] != environment['versions']:
    raise ValueError('Restore the saved baseline numerical/API environment before inference/comparison')
if ADAPTIVE_SCORES:
    adaptive_path = Path(ADAPTIVE_SCORES)
else:
    candidates = sorted(INPUT_ROOT.rglob('heldout_adaptive_scores.parquet'))
    if len(candidates) != 1:
        raise ValueError('Set ADAPTIVE_SCORES explicitly: '+str(candidates))
    adaptive_path = candidates[0]
adaptive_scores = pd.read_parquet(adaptive_path)
exact_match(partitions['heldout'],adaptive_scores,['adaptive_score'])
if not RUN_ID or Path(RUN_ID).name != RUN_ID or RUN_ID in {'.','..'}:
    raise ValueError('RUN_ID must be a directory name')
if OUT.resolve().is_relative_to(INPUT_ROOT.resolve()):
    raise ValueError('OUT must be writable and separate from inputs')
OUT.mkdir(parents=True,exist_ok=True)
if RESUME_FROM:
    restore_run(RESUME_FROM,OUT)
source_cfg = copy.deepcopy(base_cfg)
source_cfg['resources'].update(data_access=SOURCE_ACCESS,text_cache_gib=.1,
    session_budget_minutes=SESSION_BUDGET_MINUTES,disk_reserve_gib=DISK_RESERVE_GIB)
monitor = Monitor(source_cfg,OUT)
monitor.guard()
print('Selected:',{k:len(v) for k,v in partitions.items()})

## Pin tokenizer, context, span and score configuration
Baseline `add_special_tokens=True`, right truncation, attention/special masks and offsets are preserved. Take one contiguous span of at most 128 **encoded input IDs**, without deleting/remapping internal tokens. No external context or FIM formatting. Position zero is unpredicted; special/empty-offset scoring targets are excluded but remain in context. Short spans remain in the manifest. The floor-sized bottom fraction must contain at least one eligible target; otherwise the sample is recorded invalid with no fabricated score.

For zero-based target t, the author code includes future j in `t+1 .. min(t+5,N-2)`. Thus the last input token is not used as future evidence, and the final two target positions have zero future terms. This is explicitly verified against the pinned source loop. Original and altered context means/standard deviations are computed separately. Use the author's sigma floor 1e-6; no optional score/log-probability clipping. Scalar means/variances use stable FP64 reductions rather than retaining vocabulary logits.

In [ ]:
from shared.cache_setup import ensure_model_snapshot
from transformers import AutoConfig,AutoTokenizer
cache_dir = MODEL_CACHE_DIR or base_cfg['resources']['model_cache_dir']
ensure_model_snapshot(base_cfg['model_id'],base_cfg['model_revision'],cache_dir,
    ALLOW_MODEL_DOWNLOAD,DISK_RESERVE_GIB,WORK_ROOT,OUT/'model_cache_setup.json')
options = dict(revision=base_cfg['model_revision'],cache_dir=str(cache_dir),local_files_only=MODEL_LOCAL_FILES_ONLY)
tokenizer = AutoTokenizer.from_pretrained(base_cfg['model_id'],use_fast=True,**options)
model_config = AutoConfig.from_pretrained(base_cfg['model_id'],**options)
identity = scoring_identity(base_cfg,tokenizer,model_config,source_manifest)
experiment = dict(schema='infilling-code-experiment-v1',method_name=METHOD_NAME,method_status=METHOD_STATUS,
    statistic='pinned author normalized ratio',mode=MODE,selection=selection,scoring=identity,
    source_manifest_sha256=SOURCE_MANIFEST_SHA256,
    inputs={k:hashlib.sha256(p.read_bytes()).hexdigest() for k,p in paths.items()},
    adaptive_scores_sha256=hashlib.sha256(adaptive_path.read_bytes()).hexdigest())
if (OUT/'config.json').exists() and json.loads((OUT/'config.json').read_text()) != experiment:
    raise ValueError('Execution identity changed; choose a new RUN_ID')
write_json(OUT/'config.json',experiment)
write_json(OUT/'reproducibility.json',fixed)
write_json(OUT/'research_provenance.json',json.loads((PROJECT_ROOT/'infilling_code_attack/research_provenance.json').read_text()))
write_json(OUT/'source_version_hashes.json',dict(source_manifest_sha256=SOURCE_MANIFEST_SHA256,
    sources=source_manifest,versions=environment,scoring_sha256=fingerprint(identity)))
write_json(OUT/'input_paths.json',dict(**{k:str(v) for k,v in paths.items()},adaptive_scores=str(adaptive_path)))
for role,frame in partitions.items():
    frame.to_parquet(OUT/(role+'_manifest.parquet'),index=False)
write_json(OUT/'run_manifest.json',dict(status='preparing',method_name=METHOD_NAME,statistic='pinned author normalized ratio',
    mode=MODE,selection=selection,scoring_sha256=fingerprint(identity)))
cache = Cache(OUT/'infilling_cache.sqlite')
if CACHE_REUSE_FROM:
    monitor.guard(Path(CACHE_REUSE_FROM).stat().st_size)
    cache.reuse_from(CACHE_REUSE_FROM)
source_store = SourceStore(source_cfg,OUT,monitor)
plans = prepare_plans(partitions,tokenizer,identity,cache,source_store,monitor)
print('Prepared plans:',len(plans))

## Frozen inference and atomic query checkpoints
Original statistics are computed once per unique span. Replace each target with its original causal greedy token, evaluate the true future tokens in that altered context, and recompute that context's probability mean/variance. Algebraically identical greedy=true branches require no extra query. Nonidentical altered inputs are trimmed after their last needed label, grouped by length and right-padded in batches of four with explicit attention/positions. CPU tests compare this optimization with independent prefix calls. Native decoder + LM-head chunks bound transient logits on T4; all calls use `torch.inference_mode()`.

Only sufficient statistics are persisted: true/greedy log probabilities, greedy IDs, mean, sigma, token scores and raw log-ratio diagnostics. Raw ratios are audit features, not a second evaluated attack. No complete vocabulary logits or hidden-state archives. Each query commits separately in SQLite; valid cache entries skip model calls, and complete plans skip source iteration. Disk/RAM/session checks stop before operations; persist OUT and resume. Forward/head counts and synchronized operation times are logged, including explicitly unknown counts after a hard kill.

In [ ]:
def scorer_factory():
    import torch
    if not torch.cuda.is_available():raise RuntimeError('Missing query statistics require Kaggle GPU')
    monitor.guard()
    random.seed(42);np.random.seed(42);torch.manual_seed(42);torch.cuda.manual_seed_all(42)
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False
    model,loaded_tokenizer = load_frozen_target(base_cfg,cache_dir,MODEL_LOCAL_FILES_ONLY)
    if hashlib.sha256(loaded_tokenizer.backend_tokenizer.to_str().encode()).hexdigest() != identity['tokenizer_sha256']:
        raise ValueError('Inference tokenizer differs from prepared span plans')
    pad_id = loaded_tokenizer.pad_token_id
    if pad_id is None:pad_id = loaded_tokenizer.eos_token_id
    if pad_id is None:pad_id = 0
    return TorchScorer(model,pad_id,identity['head_chunk_rows'],identity['min_sigma'])
write_json(OUT/'run_manifest.json',dict(status='scoring',method_name=METHOD_NAME,method_status=METHOD_STATUS,
    mode=MODE,selection=selection,scoring_sha256=fingerprint(identity),source_manifest_sha256=SOURCE_MANIFEST_SHA256))
scored,timing = run_scores(partitions,plans,identity,cache,scorer_factory,monitor,OUT)
import gc,torch
gc.collect()
if torch.cuda.is_available():torch.cuda.empty_cache()
print('Processed:',len(scored),'invalid:',int(scored.status.eq('invalid').sum()))
print(json.dumps(timing,indent=2))

## Calibration, coverage and paired comparison
Compute the deployed threshold using only valid calibration rows, empirical FPR <=1%; apply it globally without sign search. Held-out ROC thresholds are labeled diagnostics. Invalid samples stay in `infilling_scores.parquet` with reason and null score. Coverage reports processed, valid, invalid and per-language counts; all five compared methods use exactly the same explicitly valid held-out identities. All baseline/adaptive identities must exist before invalid filtering. If either valid calibration class is absent, threshold performance is unavailable instead of fabricated. No valid held-out classes means no AUC.

Full processes all 1000 saved held-out rows; smoke processes 30 and is preliminary. Both report effective target counts. Saved baselines retain their original spans; this experiment uses the fixed 128-ID span. Actual transformer/head calls, session time/example, persistent call audit and cached-result timing are reported separately.

In [ ]:
report = evaluate(scored,partitions,saved_scores,adaptive_scores,OUT,MODE,timing=timing)
monitor.guard()
cache.close()
write_json(OUT/'run_manifest.json',dict(status=report['status'],method_name=METHOD_NAME,method_status=METHOD_STATUS,
    mode=MODE,selection=selection,scoring_sha256=fingerprint(identity),source_manifest_sha256=SOURCE_MANIFEST_SHA256,
    author_commit=identity['author_commit'],input_hashes=experiment['inputs'],
    adaptive_scores_sha256=experiment['adaptive_scores_sha256'],selected_counts={k:len(v) for k,v in partitions.items()},
    coverage=report['coverage'],invalid_samples=report['invalid_samples'],timing=timing,
    span_summary=dict(short_spans=int(scored.short_span.sum()),
        fewer_than_128_eligible=int(scored.fewer_than_128_eligible_tokens.sum())),
    artifacts={p.name:file_hash(p) for p in OUT.iterdir() if p.is_file() and p.name in {
        'config.json','metrics.json','infilling_scores.parquet','matched_comparison.parquet','infilling_cache.sqlite',
        'source_version_hashes.json','threshold.json','timing.json','reproducibility.json','research_provenance.json'}}))
print(json.dumps(report,indent=2))

## Persist and resume
Save the **entire OUT directory** with Kaggle Save Version **with outputs**. It contains `config.json`, `metrics.json`, `infilling_scores.parquet`, `matched_comparison.parquet`, `run_manifest.json`, `infilling_cache.sqlite`, selected manifests, threshold, reproducibility specification, research/source/version provenance, progress, resource and call logs. Per-position sufficient statistics and traces reside in SQLite. Stop active cells before copying SQLite; preserve any rollback journal left by interruption.

Same-run resume: attach saved OUT, keep MODE/RUN_ID/science unchanged, set RESUME_FROM to that directory. Smoke → full: new full RUN_ID, RESUME_FROM empty, optionally CACHE_REUSE_FROM pointing to saved smoke `infilling_cache.sqlite`; attach full adaptive scores. Source/science changes invalidate keys, operational paths/budgets may move. Cache imports do not import historical call logs or thresholds. Calls killed before commit must be repeated; committed queries/full samples are reused.

Fidelity: this is the verified author-code normalized Infilling Score, with code-domain token eligibility, 128-ID span, precision/invalid-handling adaptations. It is not ordinary mask accuracy or a FIM completion score. Paper main equations, Appendix pseudocode and released code have notation/indexing inconsistencies; README states the pinned implementation choices. No original benchmark equivalence, bitwise GPU equivalence or measured T4 runtime is claimed before manual execution.